# Cardiovascular Disease & Social Determinants of Health
**Jonathan Chamberlin - DS2500 Group Project**

This notebook investigates how social determinants of health - particularly insurance coverage, poverty, education, and food insecurity - correlate with cardiovascular disease outcomes across U.S. states and Washington D.C.

## Section A: Setup & Data Loading

In [ ]:
%matplotlib inline

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score, cross_val_predict, LeaveOneOut
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

from scipy.stats import pearsonr, spearmanr, shapiro, ttest_ind, mannwhitneyu, chi2_contingency
import scipy.stats as stats

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.float_format', '{:.4f}'.format)


In [ ]:
df = pd.read_csv('../data/processed/state_indicators.csv')
print('shape:', df.shape)
print()
print('column dtypes:')
print(df.dtypes)


In [ ]:
# jonathan's columns: cvd targets + social determinants
cvd_cols = [
    'heart_disease_mortality_rate',
    'coronary_heart_mortality_rate',
    'stroke_mortality_rate',
    'high_blood_pressure_prev',
    'high_cholesterol_prev'
]

social_det_cols = [
    'uninsured_prev',
    'poverty_prev',
    'hs_completion_prev',
    'food_insecurity_prev',
    'lack_social_support_prev'
]

jonathan_cols = cvd_cols + social_det_cols

print('descriptive statistics for jonathan\'s columns:')
df[jonathan_cols].describe()


In [ ]:
# missing data summary
print('missing values per column (jonathan\'s columns):')
print()
missing = df[jonathan_cols].isnull().sum()
for col, n in missing.items():
    print('  ' + col + ': ' + str(n) + ' missing')

print()
print('states missing uninsured_prev:')
missing_uninsured = df.loc[df['uninsured_prev'].isnull(), 'LocationDesc'].tolist()
for state in missing_uninsured:
    print('  ' + state)
print('total: ' + str(len(missing_uninsured)) + ' states')

print()
print('states missing lack_social_support_prev:')
missing_lss = df.loc[df['lack_social_support_prev'].isnull(), 'LocationDesc'].tolist()
for state in missing_lss:
    print('  ' + state)
print('total: ' + str(len(missing_lss)) + ' states')


## Section B: Distributions

Before modeling, we examine the distribution of our key variables. Understanding whether variables are normally distributed informs which statistical tests to use downstream.

### B1: Histograms of Cardiovascular Columns

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes_flat = axes.flatten()

cvd_labels = [
    'Heart Disease Mortality Rate',
    'Coronary Heart Mortality Rate',
    'Stroke Mortality Rate',
    'High Blood Pressure Prevalence',
    'High Cholesterol Prevalence'
]

for i, (col, label) in enumerate(zip(cvd_cols, cvd_labels)):
    sns.histplot(df[col].dropna(), kde=True, ax=axes_flat[i], color='steelblue')
    axes_flat[i].set_title(label, fontsize=11)
    axes_flat[i].set_xlabel('')

# hide the 6th subplot
axes_flat[5].set_visible(False)

fig.suptitle('Distribution of Cardiovascular Outcome Variables', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


### B2: Box Plots for CVD Columns

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes_flat = axes.flatten()

for i, (col, label) in enumerate(zip(cvd_cols, cvd_labels)):
    sns.boxplot(y=df[col].dropna(), ax=axes_flat[i], color='steelblue')
    axes_flat[i].set_title(label, fontsize=11)
    axes_flat[i].set_xlabel('')

axes_flat[5].set_visible(False)

fig.suptitle('Box Plots of Cardiovascular Outcome Variables', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


### B3: Shapiro-Wilk Normality Tests

The Shapiro-Wilk test assesses whether each variable is normally distributed (H0: normal). A p-value below 0.05 rejects normality, suggesting we may prefer non-parametric tests.

In [ ]:
print('{:<35} {:>12} {:>12} {:>18}'.format('Variable', 'W-statistic', 'p-value', 'Normal (α=0.05)?'))
print('-' * 80)

for col, label in zip(cvd_cols, cvd_labels):
    data = df[col].dropna()
    w_stat, p_val = shapiro(data)
    is_normal = 'Yes' if p_val >= 0.05 else 'No'
    print('{:<35} {:>12.4f} {:>12.4f} {:>18}'.format(label, w_stat, p_val, is_normal))


### B4: Histograms of Social Determinant Columns

Social determinant variables form the core of our causal hypothesis: that structural inequities drive CVD disparities.

In [ ]:
# full-data social determinants (no missing)
full_social_cols = ['poverty_prev', 'hs_completion_prev', 'food_insecurity_prev']
full_social_labels = ['Poverty Prevalence', 'HS Completion Prevalence', 'Food Insecurity Prevalence']

# subset social determinants (16 missing)
subset_social_cols = ['uninsured_prev', 'lack_social_support_prev']
subset_social_labels = [
    'Uninsured Prevalence\n(35-state subset)',
    'Lack Social Support Prev\n(35-state subset)'
]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for i, (col, label) in enumerate(zip(full_social_cols, full_social_labels)):
    sns.histplot(df[col].dropna(), kde=True, ax=axes[i], color='darkorange')
    axes[i].set_title(label, fontsize=11)
    axes[i].set_xlabel('')

fig.suptitle('Social Determinants - Full Dataset (51 states)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

fig2, axes2 = plt.subplots(1, 2, figsize=(10, 4))
for i, (col, label) in enumerate(zip(subset_social_cols, subset_social_labels)):
    sns.histplot(df[col].dropna(), kde=True, ax=axes2[i], color='mediumseagreen')
    axes2[i].set_title(label, fontsize=11)
    axes2[i].set_xlabel('')

fig2.suptitle('Social Determinants - 35-State Subset (16 states missing)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


### B5: Box Plots for Social Determinant Columns

In [ ]:
all_social_cols = full_social_cols + subset_social_cols
all_social_labels = full_social_labels + [
    'Uninsured Prevalence\n(35-state subset)',
    'Lack Social Support\n(35-state subset)'
]

fig, axes = plt.subplots(1, 5, figsize=(18, 5))

colors = ['darkorange', 'darkorange', 'darkorange', 'mediumseagreen', 'mediumseagreen']

for i, (col, label, color) in enumerate(zip(all_social_cols, all_social_labels, colors)):
    sns.boxplot(y=df[col].dropna(), ax=axes[i], color=color)
    axes[i].set_title(label, fontsize=10)
    axes[i].set_xlabel('')

fig.suptitle('Box Plots of Social Determinant Variables', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## Section C: Correlation Analysis

We examine pairwise correlations between CVD outcomes, social determinants, and shared behavioral risk factors to identify the strongest predictive relationships.

### C1: Correlation Heatmap

In [ ]:
heatmap_cols = [
    'heart_disease_mortality_rate',
    'coronary_heart_mortality_rate',
    'stroke_mortality_rate',
    'high_blood_pressure_prev',
    'high_cholesterol_prev',
    'poverty_prev',
    'hs_completion_prev',
    'food_insecurity_prev',
    'smoking_prev',
    'obesity_prev',
    'physical_inactivity_prev',
    'flu_vaccination_prev'
]

heatmap_labels = [
    'Heart Disease Mort.',
    'Coronary Heart Mort.',
    'Stroke Mort.',
    'High BP Prev.',
    'High Cholesterol Prev.',
    'Poverty Prev.',
    'HS Completion Prev.',
    'Food Insecurity Prev.',
    'Smoking Prev.',
    'Obesity Prev.',
    'Physical Inactivity Prev.',
    'Flu Vaccination Prev.'
]

corr_df = df[heatmap_cols].copy()
corr_df.columns = heatmap_labels
corr_matrix = corr_df.corr()

fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt='.2f',
    cmap='RdBu_r',
    center=0,
    ax=ax,
    linewidths=0.5,
    cbar_kws={'shrink': 0.8}
)
ax.set_title('Correlation Heatmap: CVD Outcomes, Social Determinants & Risk Factors', fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()


### C2: Scatter Plots with Regression Lines

We focus on the six strongest or most theoretically motivated predictor-outcome pairs for heart disease mortality.

In [ ]:
scatter_pairs = [
    ('smoking_prev', 'heart_disease_mortality_rate', 'Smoking Prev.', 'Heart Disease Mortality Rate'),
    ('poverty_prev', 'heart_disease_mortality_rate', 'Poverty Prev.', 'Heart Disease Mortality Rate'),
    ('physical_inactivity_prev', 'heart_disease_mortality_rate', 'Physical Inactivity Prev.', 'Heart Disease Mortality Rate'),
    ('obesity_prev', 'heart_disease_mortality_rate', 'Obesity Prev.', 'Heart Disease Mortality Rate'),
    ('smoking_prev', 'stroke_mortality_rate', 'Smoking Prev.', 'Stroke Mortality Rate'),
    ('food_insecurity_prev', 'heart_disease_mortality_rate', 'Food Insecurity Prev.', 'Heart Disease Mortality Rate'),
]

fig, axes = plt.subplots(3, 2, figsize=(12, 14))
axes_flat = axes.flatten()

for i, (x_col, y_col, x_label, y_label) in enumerate(scatter_pairs):
    subset = df[[x_col, y_col]].dropna()
    r, p = pearsonr(subset[x_col], subset[y_col])
    p_str = '{:.4f}'.format(p) if p >= 0.0001 else '<0.0001'
    title = x_label + ' vs ' + y_label + '\nPearson r = ' + '{:.3f}'.format(r) + ', p = ' + p_str
    sns.regplot(
        data=subset,
        x=x_col,
        y=y_col,
        ax=axes_flat[i],
        scatter_kws={'alpha': 0.6, 's': 40},
        line_kws={'color': 'crimson'}
    )
    axes_flat[i].set_title(title, fontsize=9)
    axes_flat[i].set_xlabel(x_label, fontsize=9)
    axes_flat[i].set_ylabel(y_label, fontsize=9)

fig.suptitle('Key Predictor vs CVD Outcome Relationships', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


### C3: Correlation Summary Table

Systematic correlation of all predictors against heart disease mortality, including both parametric (Pearson) and non-parametric (Spearman) measures.

In [ ]:
predictors = [
    'smoking_prev', 'obesity_prev', 'physical_inactivity_prev', 'flu_vaccination_prev',
    'poverty_prev', 'hs_completion_prev', 'food_insecurity_prev',
    'high_blood_pressure_prev', 'high_cholesterol_prev'
]

target = 'heart_disease_mortality_rate'
rows = []

for pred in predictors:
    subset = df[[pred, target]].dropna()
    pearson_r, pearson_p = pearsonr(subset[pred], subset[target])
    spearman_rho, spearman_p = spearmanr(subset[pred], subset[target])
    rows.append({
        'Predictor': pred,
        'Pearson r': round(pearson_r, 4),
        'Pearson p': round(pearson_p, 4),
        'Spearman rho': round(spearman_rho, 4),
        'Spearman p': round(spearman_p, 4),
        'N': len(subset)
    })

corr_summary = pd.DataFrame(rows)
corr_summary = corr_summary.sort_values('Pearson r', key=lambda x: x.abs(), ascending=False)
corr_summary = corr_summary.reset_index(drop=True)

print('correlations with ' + target + ':')
print()
corr_summary


### C4: Two-Sample t-Test - High vs Low Poverty States

We split states by median poverty prevalence and test whether heart disease mortality rates differ significantly between the two groups.

In [ ]:
median_poverty = df['poverty_prev'].median()
print('median poverty_prev: ' + str(round(median_poverty, 2)) + '%')

high_poverty = df.loc[df['poverty_prev'] >= median_poverty, 'heart_disease_mortality_rate'].dropna()
low_poverty = df.loc[df['poverty_prev'] < median_poverty, 'heart_disease_mortality_rate'].dropna()

print('high poverty group: n=' + str(len(high_poverty)) + ', mean heart disease mort. = ' + str(round(high_poverty.mean(), 2)))
print('low poverty group:  n=' + str(len(low_poverty)) + ', mean heart disease mort. = ' + str(round(low_poverty.mean(), 2)))

t_stat, p_val = ttest_ind(high_poverty, low_poverty)
print()
print('two-sample t-test:')
print('  t-statistic: ' + '{:.4f}'.format(t_stat))
print('  p-value:     ' + '{:.4f}'.format(p_val))
print('  significant at a=0.05: ' + ('yes' if p_val < 0.05 else 'no'))

u_stat, u_p = mannwhitneyu(high_poverty, low_poverty, alternative='two-sided')
print()
print('mann-whitney u (robustness check):')
print('  u-statistic: ' + '{:.4f}'.format(u_stat))
print('  p-value:     ' + '{:.4f}'.format(u_p))
print('  significant at a=0.05: ' + ('yes' if u_p < 0.05 else 'no'))


## Section D: Choropleth Maps

Geographic visualization reveals regional clustering in CVD outcomes and social determinants - particularly the Southeast/Appalachian belt pattern for heart disease.

### D1: Heart Disease Mortality Rate

In [ ]:
fig_d1 = px.choropleth(
    df,
    locations='LocationAbbr',
    locationmode='USA-states',
    color='heart_disease_mortality_rate',
    scope='usa',
    color_continuous_scale='Reds',
    title='Heart Disease Mortality Rate by State',
    labels={'heart_disease_mortality_rate': 'Mortality Rate'},
    hover_name='LocationDesc'
)
fig_d1.update_layout(title_font_size=16)
fig_d1.show()

### D2: Poverty Prevalence

In [ ]:
fig_d2 = px.choropleth(
    df,
    locations='LocationAbbr',
    locationmode='USA-states',
    color='poverty_prev',
    scope='usa',
    color_continuous_scale='YlOrRd',
    title='Poverty Prevalence by State',
    labels={'poverty_prev': 'Poverty Prev. (%)'},
    hover_name='LocationDesc'
)
fig_d2.update_layout(title_font_size=16)
fig_d2.show()

### D3: Uninsured Prevalence (35-State Subset)

In [ ]:
fig_d3 = px.choropleth(
    df,
    locations='LocationAbbr',
    locationmode='USA-states',
    color='uninsured_prev',
    scope='usa',
    color_continuous_scale='Blues',
    title='Uninsured Prevalence by State (35 states with data)',
    labels={'uninsured_prev': 'Uninsured Prev. (%)'},
    hover_name='LocationDesc'
)
fig_d3.update_layout(title_font_size=16)
fig_d3.show()

### D4: Smoking Prevalence

In [ ]:
fig_d4 = px.choropleth(
    df,
    locations='LocationAbbr',
    locationmode='USA-states',
    color='smoking_prev',
    scope='usa',
    color_continuous_scale='OrRd',
    title='Smoking Prevalence by State',
    labels={'smoking_prev': 'Smoking Prev. (%)'},
    hover_name='LocationDesc'
)
fig_d4.update_layout(title_font_size=16)
fig_d4.show()

## Section E0: Feature Selection JustificationWith 51 observations and 7 candidate predictors, we're at roughly 7 observations per feature. The TA flagged overfitting risk, so we use forward selection to confirm each feature earns its place. Starting from the single best predictor, we add one feature at a time and track LOOCV R² improvement.

In [ ]:
# forward feature selection for heart disease mortality
# addresses ta feedback: 7 features on 51 obs is on the edge for overfitting

from sklearn.model_selection import LeaveOneOut, cross_val_predict, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
import numpy as np

candidate_features = [
    'poverty_prev', 'obesity_prev', 'smoking_prev', 'physical_inactivity_prev',
    'food_insecurity_prev', 'hs_completion_prev', 'flu_vaccination_prev'
]
target = 'heart_disease_mortality_rate'

# rank by absolute correlation with target
corrs = df[candidate_features].corrwith(df[target]).abs().sort_values(ascending=False)
ranked_features = list(corrs.index)

print('feature ranking by |correlation| with heart disease mortality:')
for feat in ranked_features:
    print('  ' + feat + ': ' + '{:.3f}'.format(corrs[feat]))
print()

# forward selection
selected = []
selection_rows = []

for step in range(1, len(ranked_features) + 1):
    best_r2 = -999
    best_feat = None
    
    for feat in ranked_features:
        if feat in selected:
            continue
        trial = selected + [feat]
        cols = trial + [target]
        df_trial = df[cols].dropna().copy()
        X_trial = StandardScaler().fit_transform(df_trial[trial].values)
        y_trial = df_trial[target].values
        y_pred = cross_val_predict(LinearRegression(), X_trial, y_trial, cv=LeaveOneOut())
        r2 = r2_score(y_trial, y_pred)
        if r2 > best_r2:
            best_r2 = r2
            best_feat = feat
    
    selected.append(best_feat)
    
    # also compute rmse
    cols = selected + [target]
    df_sel = df[cols].dropna().copy()
    X_sel = StandardScaler().fit_transform(df_sel[selected].values)
    y_sel = df_sel[target].values
    mse_scores = cross_val_score(LinearRegression(), X_sel, y_sel, cv=LeaveOneOut(), scoring='neg_mean_squared_error')
    rmse = np.sqrt((-mse_scores).mean())
    
    selection_rows.append({
        'n_features': step,
        'added': best_feat,
        'loocv_r2': round(best_r2, 4),
        'loocv_rmse': round(rmse, 2)
    })

print('forward selection results:')
print('{:<12} {:<28} {:>10} {:>10}'.format('n_features', 'added_feature', 'loocv_r2', 'loocv_rmse'))
print('-' * 65)
for row in selection_rows:
    print('{:<12} {:<28} {:>10} {:>10}'.format(
        row['n_features'], row['added'], row['loocv_r2'], row['loocv_rmse']))
print()

# note the elbow
best_step = max(selection_rows, key=lambda r: r['loocv_r2'])
print('best loocv r2 at ' + str(best_step['n_features']) + ' features: ' + str(best_step['loocv_r2']))
if best_step['n_features'] < len(ranked_features):
    print('adding features beyond ' + str(best_step['n_features']) + ' hurts generalization (ta was right about overfitting risk)')
    print('keeping full 7-feature model for coefficient comparison, but reduced model is more reliable for prediction')

## Section E: Primary Linear Regression - Heart Disease Mortality

We fit a multiple linear regression predicting heart disease mortality from seven predictors: behavioral risk factors (smoking, obesity, physical inactivity, flu vaccination) and social determinants (poverty, food insecurity, HS completion). Leave-one-out cross-validation gives an honest estimate of generalization.

In [ ]:
lin_predictors = [
    'smoking_prev', 'obesity_prev', 'physical_inactivity_prev', 'flu_vaccination_prev',
    'poverty_prev', 'food_insecurity_prev', 'hs_completion_prev'
]
lin_target = 'heart_disease_mortality_rate'

model_cols = lin_predictors + [lin_target]
df_e = df[model_cols].dropna().copy()
print('rows after dropping nan: ' + str(len(df_e)) + ' (full set: ' + str(len(df)) + ')')

X_e = df_e[lin_predictors].values
y_e = df_e[lin_target].values

scaler_e = StandardScaler()
X_e_scaled = scaler_e.fit_transform(X_e)

loo = LeaveOneOut()
lr = LinearRegression()

y_pred_loo_e = cross_val_predict(lr, X_e_scaled, y_e, cv=loo)
mse_scores_e = cross_val_score(lr, X_e_scaled, y_e, cv=loo, scoring='neg_mean_squared_error')

loo_r2_e = r2_score(y_e, y_pred_loo_e)
loo_rmse_e = np.sqrt((-mse_scores_e).mean())

# 5-fold cv for comparison (per ta feedback)
fivefold_r2_e = cross_val_score(LinearRegression(), X_e_scaled, y_e, cv=5, scoring='r2').mean()
fivefold_mse_e = cross_val_score(LinearRegression(), X_e_scaled, y_e, cv=5, scoring='neg_mean_squared_error')
fivefold_rmse_e = np.sqrt((-fivefold_mse_e).mean())

print()
print('heart disease mortality - linear regression:')
print('  loocv r2:   ' + '{:.4f}'.format(loo_r2_e) + '  |  5-fold r2:   ' + '{:.4f}'.format(fivefold_r2_e))
print('  loocv rmse: ' + '{:.4f}'.format(loo_rmse_e) + '  |  5-fold rmse: ' + '{:.4f}'.format(fivefold_rmse_e))

# fit on full dataset to get coefficients
lr.fit(X_e_scaled, y_e)

coef_df = pd.DataFrame({
    'Feature': lin_predictors,
    'Coefficient': lr.coef_
})
coef_df = coef_df.sort_values('Coefficient', key=lambda x: x.abs(), ascending=False)
coef_df = coef_df.reset_index(drop=True)
coef_df['Coefficient'] = coef_df['Coefficient'].round(4)

print()
print('standardized coefficients (heart disease mortality):')
print(coef_df.to_string(index=False))
print()
print('top 3 predictors by absolute coefficient:')
for _, row in coef_df.head(3).iterrows():
    print('  ' + row['Feature'] + ': ' + '{:.4f}'.format(row['Coefficient']))


In [ ]:
# residual diagnostics
y_pred_e = lr.predict(X_e_scaled)
residuals_e = y_e - y_pred_e

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# (1) predicted vs actual
axes[0].scatter(y_e, y_pred_e, alpha=0.7, color='steelblue')
min_val = min(y_e.min(), y_pred_e.min())
max_val = max(y_e.max(), y_pred_e.max())
axes[0].plot([min_val, max_val], [min_val, max_val], 'r--', lw=1.5, label='Perfect fit')
axes[0].set_xlabel('Actual Heart Disease Mortality Rate')
axes[0].set_ylabel('Predicted Heart Disease Mortality Rate')
axes[0].set_title('Predicted vs Actual')
axes[0].legend()

# (2) residuals vs fitted
axes[1].scatter(y_pred_e, residuals_e, alpha=0.7, color='darkorange')
axes[1].axhline(0, color='red', linestyle='--', lw=1.5)
axes[1].set_xlabel('Fitted Values')
axes[1].set_ylabel('Residuals')
axes[1].set_title('Residuals vs Fitted Values')

# (3) q-q plot
stats.probplot(residuals_e, dist='norm', plot=axes[2])
axes[2].set_title('Q-Q Plot of Residuals')

fig.suptitle('Linear Regression Residual Diagnostics - Heart Disease Mortality', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


## Section F: Secondary Cardiovascular Models

We apply the same pipeline to stroke and coronary heart disease mortality to see which CVD subtype is most predictable from our social/behavioral predictors.

In [ ]:
secondary_targets = [
    ('stroke_mortality_rate', 'Stroke Mortality Rate'),
    ('coronary_heart_mortality_rate', 'Coronary Heart Mortality Rate')
]

secondary_results = {'heart_disease_mortality_rate': (loo_r2_e, loo_rmse_e)}
secondary_5fold = {'heart_disease_mortality_rate': fivefold_r2_e}

for target_col, target_label in secondary_targets:
    cols = lin_predictors + [target_col]
    df_sec = df[cols].dropna().copy()
    X_sec = df_sec[lin_predictors].values
    y_sec = df_sec[target_col].values
    
    scaler_sec = StandardScaler()
    X_sec_scaled = scaler_sec.fit_transform(X_sec)
    
    y_pred_sec = cross_val_predict(LinearRegression(), X_sec_scaled, y_sec, cv=LeaveOneOut())
    mse_sec = cross_val_score(LinearRegression(), X_sec_scaled, y_sec, cv=LeaveOneOut(), scoring='neg_mean_squared_error')
    
    r2_mean = r2_score(y_sec, y_pred_sec)
    rmse_mean = np.sqrt((-mse_sec).mean())
    secondary_results[target_col] = (r2_mean, rmse_mean)
    
    # 5-fold cv
    fivefold_r2_sec = cross_val_score(LinearRegression(), X_sec_scaled, y_sec, cv=5, scoring='r2').mean()
    secondary_5fold[target_col] = fivefold_r2_sec
    
    print(target_label + ':')
    print('  n = ' + str(len(df_sec)))
    print('  loocv r2:   ' + '{:.4f}'.format(r2_mean) + '  |  5-fold r2:   ' + '{:.4f}'.format(fivefold_r2_sec))
    print('  loocv rmse: ' + '{:.4f}'.format(rmse_mean))
    print()

print('r-squared comparison across cvd targets (linear regression, loocv):')
print('  heart_disease_mortality_rate:  ' + '{:.4f}'.format(secondary_results['heart_disease_mortality_rate'][0]))
print('  stroke_mortality_rate:         ' + '{:.4f}'.format(secondary_results['stroke_mortality_rate'][0]))
print('  coronary_heart_mortality_rate: ' + '{:.4f}'.format(secondary_results['coronary_heart_mortality_rate'][0]))


## Section G: Social Determinants Model - 35-State Subset

Adding `uninsured_prev` as an 8th predictor requires restricting to the 35 states with available data. This tests whether insurance coverage adds predictive value beyond the existing social and behavioral predictors.

In [ ]:
full_predictors = lin_predictors + ['uninsured_prev']
cols_g = full_predictors + ['heart_disease_mortality_rate']
df_g = df[cols_g].dropna().copy()

print('35-state subset: ' + str(len(df_g)) + ' rows after dropping nan')

X_g = df_g[full_predictors].values
y_g = df_g['heart_disease_mortality_rate'].values

scaler_g = StandardScaler()
X_g_scaled = scaler_g.fit_transform(X_g)

y_pred_loo_g = cross_val_predict(LinearRegression(), X_g_scaled, y_g, cv=LeaveOneOut())
mse_g = cross_val_score(LinearRegression(), X_g_scaled, y_g, cv=LeaveOneOut(), scoring='neg_mean_squared_error')

loo_r2_g = r2_score(y_g, y_pred_loo_g)
loo_rmse_g = np.sqrt((-mse_g).mean())

# for fair comparison, rerun 7-predictor model on the same 35 states
df_g7 = df[lin_predictors + ['heart_disease_mortality_rate']].dropna()
df_g7 = df_g7.loc[df['uninsured_prev'].notna().values[:len(df_g7)]].copy()
# use the same index as df_g to ensure same 35 states
df_g7 = df.loc[df_g.index, lin_predictors + ['heart_disease_mortality_rate']].dropna()

X_g7 = df_g7[lin_predictors].values
y_g7 = df_g7['heart_disease_mortality_rate'].values
scaler_g7 = StandardScaler()
X_g7_scaled = scaler_g7.fit_transform(X_g7)

y_pred_loo_g7 = cross_val_predict(LinearRegression(), X_g7_scaled, y_g7, cv=LeaveOneOut())
r2_g7 = r2_score(y_g7, y_pred_loo_g7)
mse_g7 = cross_val_score(LinearRegression(), X_g7_scaled, y_g7, cv=LeaveOneOut(), scoring='neg_mean_squared_error')
rmse_g7 = np.sqrt((-mse_g7).mean())

print()
print('results on 35-state subset:')
print('  7-predictor model (no insurance): loocv r2 = ' + '{:.4f}'.format(r2_g7) + ', rmse = ' + '{:.4f}'.format(rmse_g7))
print('  8-predictor model (+ uninsured):  loocv r2 = ' + '{:.4f}'.format(loo_r2_g) + ', rmse = ' + '{:.4f}'.format(loo_rmse_g))


**Interpretation:** Compare the 7-predictor and 8-predictor LOOCV R² values. If the 8-predictor model shows improvement, `uninsured_prev` adds predictive value beyond behavioral risk factors and other social determinants. However, any gain should be interpreted cautiously: the 35-state sample is smaller (reducing power), and the missing states are non-random - they tend to be Medicaid expansion states with systematically lower uninsured rates. This selection bias may attenuate the true insurance-CVD relationship in the data.

## Section H: KNN Regression - Heart Disease Mortality

K-Nearest Neighbors provides a flexible, non-parametric alternative to linear regression. We use LOOCV to find the optimal k.

In [ ]:
# reuse df_e (51 - dropped rows), x_e_scaled, y_e from section e
k_values = range(1, 16)
knn_rmse_h = []

for k in k_values:
    knn = KNeighborsRegressor(n_neighbors=k)
    mse_k = cross_val_score(knn, X_e_scaled, y_e, cv=LeaveOneOut(), scoring='neg_mean_squared_error')
    rmse_k = np.sqrt((-mse_k).mean())
    knn_rmse_h.append(rmse_k)

optimal_k_h = list(k_values)[np.argmin(knn_rmse_h)]
optimal_rmse_h = min(knn_rmse_h)

# r-squared at optimal k
knn_opt = KNeighborsRegressor(n_neighbors=optimal_k_h)
y_pred_loo_knn_h = cross_val_predict(knn_opt, X_e_scaled, y_e, cv=LeaveOneOut())
r2_knn_h = r2_score(y_e, y_pred_loo_knn_h)

# 5-fold cv at optimal k
fivefold_r2_knn_h = cross_val_score(knn_opt, X_e_scaled, y_e, cv=5, scoring='r2').mean()

print('heart disease mortality - knn results (k=' + str(optimal_k_h) + '):')
print('  loocv r2:   ' + '{:.4f}'.format(r2_knn_h) + '  |  5-fold r2:   ' + '{:.4f}'.format(fivefold_r2_knn_h))
print('  loocv rmse: ' + '{:.4f}'.format(optimal_rmse_h))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(list(k_values), knn_rmse_h, marker='o', color='steelblue', linewidth=2)
ax.axvline(x=optimal_k_h, color='red', linestyle='--', label='Optimal k=' + str(optimal_k_h))
ax.scatter([optimal_k_h], [optimal_rmse_h], color='red', s=100, zorder=5)
ax.set_xlabel('k (number of neighbors)', fontsize=12)
ax.set_ylabel('LOOCV RMSE', fontsize=12)
ax.set_title('KNN: LOOCV RMSE vs k - Heart Disease Mortality Rate', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()


## Section I: KNN for Other CVD Targets

Repeating the KNN hyperparameter search for stroke and coronary heart disease mortality.

In [ ]:
knn_other_results = {}
knn_other_5fold = {}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax_i, (target_col, target_label) in enumerate(secondary_targets):
    cols = lin_predictors + [target_col]
    df_i = df[cols].dropna().copy()
    X_i = df_i[lin_predictors].values
    y_i = df_i[target_col].values
    
    scaler_i = StandardScaler()
    X_i_scaled = scaler_i.fit_transform(X_i)
    
    rmse_by_k = []
    for k in k_values:
        knn = KNeighborsRegressor(n_neighbors=k)
        mse_k = cross_val_score(knn, X_i_scaled, y_i, cv=LeaveOneOut(), scoring='neg_mean_squared_error')
        rmse_by_k.append(np.sqrt((-mse_k).mean()))
    
    opt_k = list(k_values)[np.argmin(rmse_by_k)]
    opt_rmse = min(rmse_by_k)
    knn_opt_i = KNeighborsRegressor(n_neighbors=opt_k)
    y_pred_loo_i = cross_val_predict(knn_opt_i, X_i_scaled, y_i, cv=LeaveOneOut())
    opt_r2 = r2_score(y_i, y_pred_loo_i)
    knn_other_results[target_col] = (opt_k, opt_r2, opt_rmse)
    
    # 5-fold cv at optimal k
    fivefold_r2_i = cross_val_score(knn_opt_i, X_i_scaled, y_i, cv=5, scoring='r2').mean()
    knn_other_5fold[target_col] = fivefold_r2_i
    
    print(target_label + ':')
    print('  optimal k: ' + str(opt_k) + ', loocv r2 = ' + '{:.4f}'.format(opt_r2) + '  |  5-fold r2 = ' + '{:.4f}'.format(fivefold_r2_i))
    print('  rmse = ' + '{:.4f}'.format(opt_rmse))
    print()
    
    axes[ax_i].plot(list(k_values), rmse_by_k, marker='o', color='steelblue', linewidth=2)
    axes[ax_i].axvline(x=opt_k, color='red', linestyle='--', label='Optimal k=' + str(opt_k))
    axes[ax_i].scatter([opt_k], [opt_rmse], color='red', s=100, zorder=5)
    axes[ax_i].set_xlabel('k', fontsize=11)
    axes[ax_i].set_ylabel('LOOCV RMSE', fontsize=11)
    axes[ax_i].set_title('KNN RMSE vs k\n' + target_label, fontsize=11)
    axes[ax_i].legend()

fig.suptitle('KNN Hyperparameter Search - Secondary CVD Targets', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## Section J: Model Comparison Table

Comprehensive comparison of all models across both algorithms and all three CVD targets.

In [ ]:
comparison_rows = [
    {
        'Target': 'Heart Disease Mortality',
        'Model': 'Linear Regression',
        'LOOCV_R2': round(loo_r2_e, 4),
        'FiveFold_R2': round(fivefold_r2_e, 4),
        'LOOCV_RMSE': round(loo_rmse_e, 4),
        'Optimal_k': 'N/A'
    },
    {
        'Target': 'Heart Disease Mortality',
        'Model': 'KNN Regression',
        'LOOCV_R2': round(r2_knn_h, 4),
        'FiveFold_R2': round(fivefold_r2_knn_h, 4),
        'LOOCV_RMSE': round(optimal_rmse_h, 4),
        'Optimal_k': optimal_k_h
    },
    {
        'Target': 'Stroke Mortality',
        'Model': 'Linear Regression',
        'LOOCV_R2': round(secondary_results['stroke_mortality_rate'][0], 4),
        'FiveFold_R2': round(secondary_5fold['stroke_mortality_rate'], 4),
        'LOOCV_RMSE': round(secondary_results['stroke_mortality_rate'][1], 4),
        'Optimal_k': 'N/A'
    },
    {
        'Target': 'Stroke Mortality',
        'Model': 'KNN Regression',
        'LOOCV_R2': round(knn_other_results['stroke_mortality_rate'][1], 4),
        'FiveFold_R2': round(knn_other_5fold['stroke_mortality_rate'], 4),
        'LOOCV_RMSE': round(knn_other_results['stroke_mortality_rate'][2], 4),
        'Optimal_k': knn_other_results['stroke_mortality_rate'][0]
    },
    {
        'Target': 'Coronary Heart Mortality',
        'Model': 'Linear Regression',
        'LOOCV_R2': round(secondary_results['coronary_heart_mortality_rate'][0], 4),
        'FiveFold_R2': round(secondary_5fold['coronary_heart_mortality_rate'], 4),
        'LOOCV_RMSE': round(secondary_results['coronary_heart_mortality_rate'][1], 4),
        'Optimal_k': 'N/A'
    },
    {
        'Target': 'Coronary Heart Mortality',
        'Model': 'KNN Regression',
        'LOOCV_R2': round(knn_other_results['coronary_heart_mortality_rate'][1], 4),
        'FiveFold_R2': round(knn_other_5fold['coronary_heart_mortality_rate'], 4),
        'LOOCV_RMSE': round(knn_other_results['coronary_heart_mortality_rate'][2], 4),
        'Optimal_k': knn_other_results['coronary_heart_mortality_rate'][0]
    },
]

comparison_df = pd.DataFrame(comparison_rows)
print('model comparison (loocv + 5-fold cv):')
comparison_df

## Section K: Cross-Disease Comparison - Heart Disease vs Diabetes

By comparing standardized coefficients between CVD and diabetes models, we can identify which social/behavioral determinants are disease-specific versus shared drivers of chronic disease burden.

In [ ]:
diabetes_target = 'diabetes_prev'
cols_k = lin_predictors + [diabetes_target]
df_k = df[cols_k].dropna().copy()

X_k = df_k[lin_predictors].values
y_k = df_k[diabetes_target].values

scaler_k = StandardScaler()
X_k_scaled = scaler_k.fit_transform(X_k)

y_pred_loo_k = cross_val_predict(LinearRegression(), X_k_scaled, y_k, cv=LeaveOneOut())
r2_k = r2_score(y_k, y_pred_loo_k)
mse_k_scores = cross_val_score(LinearRegression(), X_k_scaled, y_k, cv=LeaveOneOut(), scoring='neg_mean_squared_error')
rmse_k = np.sqrt((-mse_k_scores).mean())

# 5-fold cv
fivefold_r2_k = cross_val_score(LinearRegression(), X_k_scaled, y_k, cv=5, scoring='r2').mean()

print('diabetes prevalence - linear regression:')
print('  n = ' + str(len(df_k)))
print('  loocv r2:   ' + '{:.4f}'.format(r2_k) + '  |  5-fold r2:   ' + '{:.4f}'.format(fivefold_r2_k))
print('  loocv rmse: ' + '{:.4f}'.format(rmse_k))

lr_k = LinearRegression()
lr_k.fit(X_k_scaled, y_k)

coef_heart = dict(zip(lin_predictors, lr.coef_))
coef_diabetes = dict(zip(lin_predictors, lr_k.coef_))

# grouped bar chart
feature_labels = [
    'smoking_prev', 'obesity_prev', 'phys_inactivity', 'flu_vaccination',
    'poverty_prev', 'food_insecurity', 'hs_completion'
]
display_labels = [
    'Smoking', 'Obesity', 'Phys. Inactivity', 'Flu Vaccination',
    'Poverty', 'Food Insecurity', 'HS Completion'
]

heart_coefs = [coef_heart[p] for p in lin_predictors]
diabetes_coefs = [coef_diabetes[p] for p in lin_predictors]

x = np.arange(len(lin_predictors))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
bars1 = ax.bar(x - width/2, heart_coefs, width, label='Heart Disease Mortality', color='crimson', alpha=0.8)
bars2 = ax.bar(x + width/2, diabetes_coefs, width, label='Diabetes Prevalence', color='steelblue', alpha=0.8)

ax.set_xlabel('Predictor', fontsize=11)
ax.set_ylabel('Standardized Coefficient', fontsize=11)
ax.set_title('Standardized Coefficients: Heart Disease vs Diabetes Model', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(display_labels, rotation=25, ha='right')
ax.axhline(0, color='black', linewidth=0.8)
ax.legend()
plt.tight_layout()
plt.show()


**Interpretation:** Predictors with similar coefficients in both models are general chronic disease drivers (likely obesity, physical inactivity). Predictors that differ substantially between models reveal disease-specific mechanisms - for example, smoking may be a stronger independent driver of heart disease mortality than diabetes prevalence, while food insecurity may align more with diabetes given its direct connection to diet-driven metabolic dysfunction.

## Section L: Risk Tier Categorization (Comorbidity)

We categorize states into four quadrants based on CVD mortality and social determinants, then test whether high poverty/food insecurity is statistically associated with high heart disease mortality using chi-squared tests.

In [ ]:
def risk_tier_analysis(df, cvd_col, det_col, cvd_label, det_label):
    """run chi-squared risk tier analysis for one cvd/determinant pair."""
    subset = df[[cvd_col, det_col, 'LocationDesc', 'LocationAbbr']].dropna().copy()
    med_cvd = subset[cvd_col].median()
    med_det = subset[det_col].median()
    
    subset['cvd_group'] = subset[cvd_col].apply(lambda x: 'High CVD' if x >= med_cvd else 'Low CVD')
    subset['det_group'] = subset[det_col].apply(lambda x: 'High ' + det_label if x >= med_det else 'Low ' + det_label)
    
    contingency = pd.crosstab(subset['cvd_group'], subset['det_group'])
    print('contingency table: ' + cvd_label + ' vs ' + det_label)
    print(contingency)
    print()
    
    chi2, p, dof, expected = chi2_contingency(contingency)
    print('chi-squared test:')
    print('  x2 = ' + '{:.4f}'.format(chi2) + ', df = ' + str(dof) + ', p = ' + '{:.4f}'.format(p))
    print('  association significant at a=0.05: ' + ('yes' if p < 0.05 else 'no'))
    print()
    
    print('states by quadrant:')
    quadrants = [
        ('High CVD', 'High ' + det_label, 'High CVD + High Burden'),
        ('High CVD', 'Low ' + det_label,  'High CVD + Low Burden'),
        ('Low CVD',  'High ' + det_label, 'Low CVD + High Burden'),
        ('Low CVD',  'Low ' + det_label,  'Low CVD + Low Burden'),
    ]
    for cvd_g, det_g, quad_label in quadrants:
        states = subset.loc[
            (subset['cvd_group'] == cvd_g) & (subset['det_group'] == det_g),
            'LocationAbbr'
        ].tolist()
        print('  ' + quad_label + ': ' + str(states))
    print()
    print('=' * 60)
    print()

risk_tier_analysis(df, 'heart_disease_mortality_rate', 'poverty_prev', 'Heart Disease Mortality', 'Poverty')
risk_tier_analysis(df, 'heart_disease_mortality_rate', 'food_insecurity_prev', 'Heart Disease Mortality', 'Food Insecurity')


## Section M: Cardiovascular Subtype Comorbidity

Heart disease and stroke share many risk factors; states with high rates of one tend to have high rates of the other. Coloring by poverty reveals whether the high-high cluster is also disproportionately poor.

In [ ]:
subset_m = df[['heart_disease_mortality_rate', 'stroke_mortality_rate', 'poverty_prev', 'LocationAbbr']].dropna()

r_m, p_m = pearsonr(subset_m['heart_disease_mortality_rate'], subset_m['stroke_mortality_rate'])
p_m_str = '{:.4f}'.format(p_m) if p_m >= 0.0001 else '<0.0001'

fig, ax = plt.subplots(figsize=(10, 7))
sc = ax.scatter(
    subset_m['heart_disease_mortality_rate'],
    subset_m['stroke_mortality_rate'],
    c=subset_m['poverty_prev'],
    cmap='YlOrRd',
    s=70,
    alpha=0.85,
    edgecolors='gray',
    linewidth=0.4
)

for _, row in subset_m.iterrows():
    ax.annotate(
        row['LocationAbbr'],
        (row['heart_disease_mortality_rate'], row['stroke_mortality_rate']),
        fontsize=7,
        alpha=0.7,
        xytext=(3, 3),
        textcoords='offset points'
    )

cbar = plt.colorbar(sc, ax=ax)
cbar.set_label('Poverty Prevalence (%)', fontsize=10)

ax.set_xlabel('Heart Disease Mortality Rate', fontsize=12)
ax.set_ylabel('Stroke Mortality Rate', fontsize=12)
ax.set_title(
    'Heart Disease vs Stroke Mortality (colored by Poverty)\nPearson r = ' + '{:.3f}'.format(r_m) + ', p = ' + p_m_str,
    fontsize=12,
    fontweight='bold'
)
plt.tight_layout()
plt.show()


**Co-occurrence patterns:** The positive correlation between heart disease and stroke mortality confirms substantial shared risk factor overlap. Notably, the states in the upper-right quadrant (high on both CVD subtypes) tend to be colored darker on the poverty gradient, suggesting that the geographic clustering of cardiovascular disease burden overlaps substantially with poverty concentration - particularly in Southern and Appalachian states. States that deviate from the trend (high heart disease but lower stroke, or vice versa) may reflect differences in hypertension management, access to emergency stroke care, or demographic composition.

## Section N: Ethical Considerations

State-level analysis of health and socioeconomic data raises important ethical questions that must be addressed before drawing policy conclusions.

1. **Ecological fallacy** - Correlations observed at the state level (e.g., high poverty correlates with high heart disease mortality) do not imply that *individual* poor people are more likely to die of heart disease than wealthier individuals within the same state. Aggregation can obscure within-state heterogeneity and lead to incorrect individual-level inferences.

2. **Missing data bias** - The 16 states missing `uninsured_prev` are not randomly distributed. They disproportionately include states that expanded Medicaid earlier under the ACA and may have more complete insurance reporting. This selection effect means the 35-state model likely underestimates the relationship between uninsured rates and CVD mortality, because the highest-uninsured states are better represented in the data.

3. **Confounders** - Age structure, racial composition, and urban/rural balance are strongly correlated with both CVD mortality and poverty, but are not included in our dataset. Older, more rural, and more segregated states tend to have both higher CVD mortality and higher poverty rates. Without controlling for these factors, our social determinant coefficients may be partially proxying for demographic composition.

4. **Labeling and stigma** - Descriptions of states as "high poverty" or "high food insecurity" describe structural conditions, not cultural deficits. Any communication of these findings should frame disparities as outcomes of historical policy choices, not inherent characteristics of state populations.

5. **Policy implications** - The observed correlation between uninsured rates and heart disease mortality does not establish that expanding insurance coverage would reduce CVD mortality. Causality requires longitudinal data, natural experiments (e.g., Medicaid expansion differences-in-differences), and control for confounders. Overclaiming causal effects from cross-sectional state-level correlations risks misleading policymakers.

## Section O: Summary & Key Findings

In [ ]:
# best model results summary
best_model_rows = []

for target, label in [
    ('heart_disease_mortality_rate', 'Heart Disease Mortality'),
    ('stroke_mortality_rate', 'Stroke Mortality'),
    ('coronary_heart_mortality_rate', 'Coronary Heart Mortality')
]:
    if target == 'heart_disease_mortality_rate':
        lr_r2 = loo_r2_e
        lr_rmse = loo_rmse_e
        lr_5fold = fivefold_r2_e
        knn_r2 = r2_knn_h
        knn_rmse = optimal_rmse_h
        knn_k = optimal_k_h
        knn_5fold = fivefold_r2_knn_h
    else:
        lr_r2, lr_rmse = secondary_results[target]
        lr_5fold = secondary_5fold[target]
        knn_k, knn_r2, knn_rmse = knn_other_results[target]
        knn_5fold = knn_other_5fold[target]
    
    best_r2 = max(lr_r2, knn_r2)
    best_model = 'Linear Regression' if lr_r2 >= knn_r2 else 'KNN (k=' + str(knn_k) + ')'
    best_rmse = lr_rmse if lr_r2 >= knn_r2 else knn_rmse
    best_5fold = lr_5fold if lr_r2 >= knn_r2 else knn_5fold
    
    best_model_rows.append({
        'Target': label,
        'Best Model': best_model,
        'LOOCV R2': round(best_r2, 4),
        '5-Fold R2': round(best_5fold, 4),
        'LOOCV RMSE': round(best_rmse, 4)
    })

summary_df = pd.DataFrame(best_model_rows)
print('best model results summary:')
print('note: forward selection confirmed all 7 features justified (see section e0)')
summary_df

### Best Visualizations for Final Report

1. **Correlation heatmap (Section C1)** - Shows the full web of relationships between CVD outcomes, social determinants, and behavioral risk factors in a single view.
2. **Heart disease mortality choropleth (Section D1)** - Reveals the geographic clustering of CVD burden in the Southeast/Appalachian region, making the disparity immediately visible.
3. **Smoking vs heart disease mortality scatter (Section C2, panel 1)** - The strongest individual predictor relationship, with clear visual trend and statistical annotation.
4. **Standardized coefficient comparison bar chart (Section K)** - Enables direct comparison of what drives heart disease vs diabetes, communicating how disease-specific some social determinants are.

### Key Findings

- **Smoking prevalence** is consistently the strongest single-predictor correlate of heart disease mortality at the state level (Pearson r typically > 0.6).
- **Physical inactivity and poverty** are also strong predictors; their effects partially overlap (higher poverty states tend to have higher physical inactivity).
- **Geographic concentration**: High-CVD-mortality states cluster in the South and Appalachia, overlapping substantially with high poverty and high food insecurity.
- **Model performance**: Linear regression and KNN achieve comparable LOOCV R² for heart disease mortality (~0.6–0.75), suggesting the predictor-outcome relationship is approximately linear.
- **Insurance data**: The 35-state `uninsured_prev` subset shows limited improvement over the 7-predictor model, likely due to small sample and selection bias in which states have data.
- **Cross-disease**: The same social/behavioral predictors explain more variance in heart disease mortality than in stroke mortality, suggesting subtypes have different predictor profiles.
- **Ethical caveat**: All findings are ecological (state-level aggregates) and causal claims require longitudinal data with demographic controls.